# 06 — Clustering Algorithm Comparison

Compare HDBSCAN, KMeans, and Agglomerative (hierarchical) clustering on the best UMAP projection from nb 05.
All results are cached and logged to `results.csv` with **three-tier quality metrics**.

**Prerequisite:** Run `05_dimensionality_reduction.ipynb` first and set the `BEST_*` variables below.

## Why three evaluation metrics instead of one?

Using silhouette score alone to compare clustering configurations is tempting but misleading for topic discovery. Here's why we use three complementary metrics:

### Tier 1 — Geometric metrics (silhouette, Davies-Bouldin, Calinski-Harabasz)

These measure how well-separated and compact clusters are *in the UMAP-reduced embedding space*.

**Problem:** Silhouette rewards geometric separation, not semantic meaningfulness. Two failure modes:
- **Sentiment collapse:** A model that creates one cluster of 5-star reviews and one of 1-star reviews scores very high silhouette (the polarity is real and geometrically clear). But this tells us nothing about *topics* — it tells us guests who love hotels write differently than guests who hate them, which is obvious and not useful.
- **Instruction model inflation:** Instruction-tuned embeddings like `instructor-large` with a domain prompt literally reshape the geometry to make clusters easier to separate. Their silhouette is mechanically higher, not because the topics are better, but because the embedding space was pushed toward the clustering goal. An independent check is needed.

### Tier 2 — Topic coherence C_v (Gensim, independent of embedding geometry)

C_v coherence asks: do the top words of each cluster co-occur in the actual text corpus? It uses normalised pointwise mutual information (NPMI) across a sliding window.

**Why it matters:** Computed purely from word co-occurrence, not embeddings. A cluster that looks geometrically tight but mixes topics (e.g. 'great staff' + 'nice location' crammed together) will score low coherence even if its silhouette is high. High C_v confirms that each cluster has a distinct, internally consistent vocabulary.

**Scale:** 0–1, higher is better. Typical good topic models score 0.4–0.7 on C_v. Set `COMPUTE_COHERENCE = False` to skip it in fast sweep mode.

### Tier 3 — Star-rating entropy (thematic vs sentiment-blob detection)

A **thematic cluster** (e.g. 'hotel breakfast') attracts guests who loved breakfast and guests who didn't — mixed star ratings → high entropy.

A **sentiment cluster** (e.g. 'overall positive stay') attracts only high-rated reviews → low entropy (collapsed to 4–5 stars).

Entropy is normalised by log(5) so the range is [0, 1]:
- **> 0.85** → strongly thematic (topic drives the cluster, not rating)
- **0.60–0.85** → mixed
- **< 0.60** → sentiment-dominated blob

### The real winner is top-right in silhouette × coherence space

A configuration that scores high silhouette *and* high C_v *and* high rating entropy has found real, geometrically clean, lexically coherent, topic-driven clusters — not just well-separated sentiment poles.

## Setup

In [ ]:
import sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '..')

import hdbscan
from sklearn.cluster import KMeans, AgglomerativeClustering

from utils import (
    load_config, get_preprocessor,
    embedding_path, umap_path, clustering_path,
    save_array, load_array, array_exists, make_slug,
    compute_metrics, compute_coherence, compute_rating_entropy,
    log_result,
)

cfg = load_config(sample_size=5_000, device="cpu")
cfg.ensure_dirs()

# ── Set from nb 04 + 05 findings ──────────────────────────────────────────────
BEST_MODEL       = "all-mpnet-base-v2"
BEST_INSTRUCTION = "no_inst"
BEST_NC          = 10    # n_components
BEST_NN          = 15    # n_neighbors
BEST_METRIC      = "cosine"
MIN_DIST         = 0.0
# ─────────────────────────────────────────────────────────────────────────────

# Set False to skip coherence in fast iteration (saves ~30s per experiment)
COMPUTE_COHERENCE = True

# Slug used in clustering_path filenames
UMAP_SLUG = (f"{make_slug(BEST_MODEL)}__{make_slug(BEST_INSTRUCTION)}"
             f"__nc{BEST_NC}__nn{BEST_NN}")

print(f"Config: {BEST_MODEL} | {BEST_INSTRUCTION} | nc={BEST_NC} | nn={BEST_NN} | {BEST_METRIC}")
print(f"COMPUTE_COHERENCE = {COMPUTE_COHERENCE}")

In [ ]:
# Load best UMAP projection and 2-D visualisation projection
reduced = load_array(
    umap_path(cfg.cache_dir, BEST_MODEL, BEST_NC, BEST_NN, MIN_DIST,
               BEST_METRIC, cfg.sample_size, instruction=BEST_INSTRUCTION)
)
reduced_2d = load_array(
    umap_path(cfg.cache_dir, BEST_MODEL, 2, BEST_NN, 0.1,
               "cosine", cfg.sample_size, instruction=BEST_INSTRUCTION, prefix="viz_")
)

# Load texts AND star ratings (stars needed for Tier 3 entropy check)
preprocess = get_preprocessor("minimal")
texts = []
stars = []
with open(cfg.data_path) as f:
    for line in f:
        obj = json.loads(line)
        texts.append(preprocess(obj["text"]))
        stars.append(float(obj.get("stars", float("nan"))))
        if len(texts) >= cfg.sample_size:
            break

stars = np.array(stars)
print(f"reduced: {reduced.shape}  |  reduced_2d: {reduced_2d.shape}")
print(f"texts: {len(texts)}  |  stars: {stars.shape}  "
      f"(missing: {np.isnan(stars).sum()})")

## HDBSCAN — parameter sweep

`min_cluster_size` × `min_samples` grid. HDBSCAN is our primary algorithm because it:
- Does not require specifying k in advance
- Labels low-density documents as noise (-1) rather than forcing them into a cluster
- Produces variable-density clusters, which matches real topic distributions

In [ ]:
hdbscan_grid = [
    (10, 3), (10, 5),
    (15, 3), (15, 5), (15, 10),
    (20, 5), (20, 10),
    (30, 10),
]

hdbscan_results = []

for mcs, ms in hdbscan_grid:
    params_slug = f"mcs{mcs}__ms{ms}"
    path = clustering_path(cfg.cache_dir, "hdbscan", params_slug, UMAP_SLUG, cfg.sample_size)

    if array_exists(path):
        labels = load_array(path).astype(int)
        clust_time = 0.0
    else:
        t0 = time.time()
        labels = hdbscan.HDBSCAN(
            min_cluster_size=mcs, min_samples=ms
        ).fit_predict(reduced)
        clust_time = time.time() - t0
        save_array(path, labels.astype(np.int32))

    metrics = compute_metrics(reduced, labels, runtime_s=clust_time, seed=cfg.seed)

    coherence = compute_coherence(texts, labels) if COMPUTE_COHERENCE else None
    entropy   = compute_rating_entropy(stars, labels)

    hdbscan_results.append({"mcs": mcs, "ms": ms, **metrics,
                             "coherence_cv": coherence, "rating_entropy": entropy})

    log_result(cfg.results_csv, {
        "pipeline": "custom",
        "sample_size": cfg.sample_size,
        "device": cfg.device,
        "embedding_model": BEST_MODEL,
        "embedding_instruction": BEST_INSTRUCTION,
        "reduction_method": "umap",
        "umap_n_components": BEST_NC,
        "umap_n_neighbors": BEST_NN,
        "umap_min_dist": MIN_DIST,
        "umap_metric": BEST_METRIC,
        "clustering_algo": "hdbscan",
        "cluster_params": json.dumps({"min_cluster_size": mcs, "min_samples": ms}),
        **metrics,
        "coherence_cv": coherence,
        "rating_entropy": entropy,
        "notes": f"hdbscan sweep mcs={mcs} ms={ms}",
    })
    print(f"  mcs={mcs:>3d} ms={ms:>2d}  clusters={metrics['n_clusters']:>3d}  "
          f"noise={metrics['noise_ratio']:.1%}  sil={metrics['silhouette']}  "
          f"coh={coherence}  entr={entropy}")

hdbscan_df = pd.DataFrame(hdbscan_results)
hdbscan_df[["mcs", "ms", "n_clusters", "noise_ratio", "silhouette",
            "coherence_cv", "rating_entropy"]]

## KMeans — k sweep

KMeans assigns every document to a cluster (no noise label) and requires k upfront.
Useful as a baseline: if HDBSCAN and KMeans agree on good configurations, that's a stronger signal.

In [ ]:
K_RANGE = [5, 8, 10, 12, 15, 18, 20, 25, 30]
kmeans_results = []

for k in K_RANGE:
    params_slug = f"k{k}"
    path = clustering_path(cfg.cache_dir, "kmeans", params_slug, UMAP_SLUG, cfg.sample_size)

    if array_exists(path):
        labels = load_array(path).astype(int)
        clust_time = 0.0
    else:
        t0 = time.time()
        labels = KMeans(
            n_clusters=k, random_state=cfg.seed, n_init="auto"
        ).fit_predict(reduced)
        clust_time = time.time() - t0
        save_array(path, labels.astype(np.int32))

    metrics = compute_metrics(reduced, labels, runtime_s=clust_time, seed=cfg.seed)

    coherence = compute_coherence(texts, labels) if COMPUTE_COHERENCE else None
    entropy   = compute_rating_entropy(stars, labels)

    kmeans_results.append({"k": k, **metrics,
                            "coherence_cv": coherence, "rating_entropy": entropy})

    log_result(cfg.results_csv, {
        "pipeline": "custom",
        "sample_size": cfg.sample_size,
        "device": cfg.device,
        "embedding_model": BEST_MODEL,
        "embedding_instruction": BEST_INSTRUCTION,
        "reduction_method": "umap",
        "umap_n_components": BEST_NC,
        "umap_n_neighbors": BEST_NN,
        "umap_min_dist": MIN_DIST,
        "umap_metric": BEST_METRIC,
        "clustering_algo": "kmeans",
        "cluster_params": json.dumps({"k": k}),
        **metrics,
        "coherence_cv": coherence,
        "rating_entropy": entropy,
        "notes": f"kmeans sweep k={k}",
    })
    print(f"  k={k:>3d}  sil={metrics['silhouette']}  "
          f"coh={coherence}  entr={entropy}")

kmeans_df = pd.DataFrame(kmeans_results)

In [ ]:
# Silhouette + coherence vs k
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(kmeans_df["k"], kmeans_df["silhouette"], "o-", color="steelblue")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Silhouette")
axes[0].set_title("KMeans: silhouette vs k")

axes[1].plot(kmeans_df["k"], kmeans_df["calinski_harabasz"], "s-", color="coral")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Calinski-Harabasz")
axes[1].set_title("KMeans: Calinski-Harabasz vs k (elbow)")

if kmeans_df["coherence_cv"].notna().any():
    axes[2].plot(kmeans_df["k"], kmeans_df["coherence_cv"], "^-", color="seagreen")
    axes[2].set_title("KMeans: C_v coherence vs k (higher = more distinct topics)")
else:
    axes[2].set_title("Coherence not computed (COMPUTE_COHERENCE=False)")
axes[2].set_xlabel("k"); axes[2].set_ylabel("C_v coherence")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Agglomerative / Hierarchical clustering

In [ ]:
AGG_LINKAGES  = ["ward", "average", "complete"]
AGG_K_VALUES  = [10, 15, 20]
agg_results   = []

for linkage in AGG_LINKAGES:
    for k in AGG_K_VALUES:
        params_slug = f"k{k}__{linkage}"
        path = clustering_path(cfg.cache_dir, "agg", params_slug, UMAP_SLUG, cfg.sample_size)

        if array_exists(path):
            labels = load_array(path).astype(int)
            clust_time = 0.0
        else:
            t0 = time.time()
            labels = AgglomerativeClustering(
                n_clusters=k, linkage=linkage
            ).fit_predict(reduced)
            clust_time = time.time() - t0
            save_array(path, labels.astype(np.int32))

        metrics = compute_metrics(reduced, labels, runtime_s=clust_time, seed=cfg.seed)

        coherence = compute_coherence(texts, labels) if COMPUTE_COHERENCE else None
        entropy   = compute_rating_entropy(stars, labels)

        agg_results.append({"linkage": linkage, "k": k, **metrics,
                             "coherence_cv": coherence, "rating_entropy": entropy})

        log_result(cfg.results_csv, {
            "pipeline": "custom",
            "sample_size": cfg.sample_size,
            "device": cfg.device,
            "embedding_model": BEST_MODEL,
            "embedding_instruction": BEST_INSTRUCTION,
            "reduction_method": "umap",
            "umap_n_components": BEST_NC,
            "umap_n_neighbors": BEST_NN,
            "umap_min_dist": MIN_DIST,
            "umap_metric": BEST_METRIC,
            "clustering_algo": "agglomerative",
            "cluster_params": json.dumps({"k": k, "linkage": linkage}),
            **metrics,
            "coherence_cv": coherence,
            "rating_entropy": entropy,
            "notes": f"agg linkage={linkage} k={k}",
        })
        print(f"  linkage={linkage:<8s} k={k:>3d}  sil={metrics['silhouette']}  "
              f"coh={coherence}  entr={entropy}")

agg_df = pd.DataFrame(agg_results)
agg_df.pivot(index="k", columns="linkage", values="silhouette")

### Dendrogram (ward linkage, subsample)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage as sp_linkage

# Subsample for readability — dendrogram on full 5k is huge
rng = np.random.default_rng(cfg.seed)
idx = rng.choice(len(reduced), size=min(300, len(reduced)), replace=False)
sample = reduced[idx]

Z = sp_linkage(sample, method="ward")

plt.figure(figsize=(14, 4))
dendrogram(Z, no_labels=True, truncate_mode="lastp", p=30,
           color_threshold=0.7 * max(Z[:, 2]))
plt.title("Dendrogram (ward, 300-doc sample of UMAP-reduced space)")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

## Visual comparison — 2-D scatter

In [ ]:
# Update these after reviewing the metric tables above
BEST_HDBSCAN_PARAMS = (15, 5)       # (mcs, ms)
BEST_KMEANS_K       = 15
BEST_AGG_PARAMS     = (15, "ward")  # (k, linkage)

labels_hdbscan = load_array(
    clustering_path(cfg.cache_dir, "hdbscan",
                    f"mcs{BEST_HDBSCAN_PARAMS[0]}__ms{BEST_HDBSCAN_PARAMS[1]}",
                    UMAP_SLUG, cfg.sample_size)
).astype(int)

labels_kmeans = load_array(
    clustering_path(cfg.cache_dir, "kmeans", f"k{BEST_KMEANS_K}",
                    UMAP_SLUG, cfg.sample_size)
).astype(int)

labels_agg = load_array(
    clustering_path(cfg.cache_dir, "agg",
                    f"k{BEST_AGG_PARAMS[0]}__{BEST_AGG_PARAMS[1]}",
                    UMAP_SLUG, cfg.sample_size)
).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = [f"HDBSCAN (mcs={BEST_HDBSCAN_PARAMS[0]}, ms={BEST_HDBSCAN_PARAMS[1]})",
          f"KMeans (k={BEST_KMEANS_K})",
          f"Agglomerative (k={BEST_AGG_PARAMS[0]}, {BEST_AGG_PARAMS[1]})"]

for ax, labels, title in zip(axes, [labels_hdbscan, labels_kmeans, labels_agg], titles):
    noise = labels == -1
    ax.scatter(reduced_2d[noise, 0], reduced_2d[noise, 1],
               s=2, c="lightgrey", alpha=0.3, label="noise")
    ax.scatter(reduced_2d[~noise, 0], reduced_2d[~noise, 1],
               s=2, c=labels[~noise], cmap="tab20", alpha=0.6)
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Clustering comparison — 2-D UMAP projection", y=1.01)
plt.tight_layout()
plt.show()

## Qualitative inspection — sample reviews per cluster

Always read actual reviews from each cluster before trusting the numbers. Metrics can lie; text doesn't.

In [ ]:
N_CLUSTERS_INSPECT = 6
N_SAMPLES_EACH     = 3

unique_labels = sorted(set(labels_hdbscan))
unique_labels = [l for l in unique_labels if l != -1][:N_CLUSTERS_INSPECT]

for cid in unique_labels:
    mask = labels_hdbscan == cid
    members = [texts[i] for i, m in enumerate(mask) if m]
    member_stars = stars[mask]
    avg_star = member_stars[~np.isnan(member_stars)].mean() if not np.all(np.isnan(member_stars)) else float("nan")
    print(f"\n── Cluster {cid} ({sum(mask)} docs, avg stars={avg_star:.1f}) ──")
    for t in members[:N_SAMPLES_EACH]:
        print(f"  • {t[:120]}")

## Three-tier summary table

This table is the core decision aid. Look for configurations that score well across **all three tiers**, not just silhouette.

In [ ]:
# Combine all algorithms into one summary
summary_rows = []

for _, row in hdbscan_df.iterrows():
    summary_rows.append({
        "algo": "hdbscan",
        "params": f"mcs={int(row['mcs'])} ms={int(row['ms'])}",
        "n_clusters": row["n_clusters"],
        "noise_ratio": row["noise_ratio"],
        "silhouette": row["silhouette"],
        "coherence_cv": row["coherence_cv"],
        "rating_entropy": row["rating_entropy"],
    })

for _, row in kmeans_df.iterrows():
    summary_rows.append({
        "algo": "kmeans",
        "params": f"k={int(row['k'])}",
        "n_clusters": row["n_clusters"],
        "noise_ratio": 0.0,
        "silhouette": row["silhouette"],
        "coherence_cv": row["coherence_cv"],
        "rating_entropy": row["rating_entropy"],
    })

for _, row in agg_df.iterrows():
    summary_rows.append({
        "algo": "agglomerative",
        "params": f"k={int(row['k'])} {row['linkage']}",
        "n_clusters": row["n_clusters"],
        "noise_ratio": 0.0,
        "silhouette": row["silhouette"],
        "coherence_cv": row["coherence_cv"],
        "rating_entropy": row["rating_entropy"],
    })

summary_df = pd.DataFrame(summary_rows).sort_values("silhouette", ascending=False)
summary_df.head(20)

In [ ]:
# Silhouette × coherence scatter — true winners are top-right
if summary_df["coherence_cv"].notna().any():
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = {"hdbscan": "steelblue", "kmeans": "coral", "agglomerative": "seagreen"}

    for _, row in summary_df.iterrows():
        if pd.isna(row["silhouette"]) or pd.isna(row["coherence_cv"]):
            continue
        ax.scatter(row["silhouette"], row["coherence_cv"],
                   color=colors[row["algo"]], s=80, alpha=0.8)
        ax.annotate(row["params"], (row["silhouette"], row["coherence_cv"]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")

    # Add legend
    for algo, color in colors.items():
        ax.scatter([], [], color=color, label=algo, s=80)
    ax.legend()

    ax.set_xlabel("Silhouette (geometric separation)")
    ax.set_ylabel("C_v coherence (lexical distinctiveness)")
    ax.set_title("Silhouette vs C_v coherence — top-right = real winners")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Set COMPUTE_COHERENCE=True and re-run to see this plot.")

## Reproducibility check

In [ ]:
mcs, ms = BEST_HDBSCAN_PARAMS
labels_run2 = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(reduced)
assert np.array_equal(labels_hdbscan, labels_run2), "HDBSCAN is not reproducible!"
print("HDBSCAN is reproducible (identical labels on two runs)")

## Save best labels

In [ ]:
# Update BEST_ALGORITHM after reviewing the three-tier summary above
BEST_ALGORITHM = "hdbscan"     # "hdbscan" | "kmeans" | "agglomerative"

best_labels = {
    "hdbscan":       labels_hdbscan,
    "kmeans":        labels_kmeans,
    "agglomerative": labels_agg,
}[BEST_ALGORITHM]

k = cfg.sample_size // 1000
best_path = cfg.clustering_dir / f"best_labels__{k}k.npy"
save_array(best_path, best_labels.astype(np.int32))
print(f"Best algorithm: {BEST_ALGORITHM}  |  saved to {best_path.name}")

## Decision

Fill in after running. Look for the configuration that ranks well across all three tiers.

| Algorithm | Config | n_clusters | noise% | Silhouette (T1) | C_v coherence (T2) | Rating entropy (T3) | Notes |
|---|---|---|---|---|---|---|---|
| HDBSCAN | mcs=_, ms=_ | _fill_ | _fill_ | _fill_ | _fill_ | _fill_ | |
| KMeans | k=_ | _fill_ | 0% | _fill_ | _fill_ | _fill_ | |
| Agglomerative | k=_, ward | _fill_ | 0% | _fill_ | _fill_ | _fill_ | |

**Winner:** _fill in_ — chose it because:
- Silhouette ≥ _fill_  (geometric separation)
- C_v coherence ≥ _fill_  (clusters have distinct vocabularies)
- Rating entropy ≥ _fill_  (not collapsed to sentiment poles)

**Contrast case** (if applicable): config X had higher silhouette but lower coherence/entropy — likely a sentiment-dominated split, not a topic split.